<a href="https://colab.research.google.com/github/Kmjndv/VeritasAIHackathon/blob/main/Asthma_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Open the Respira app

### [Open Respira — record or upload audio](https://respira-breathe-well.smart-moss-9631.chatgpt.site)

The site is public: anyone with the link can use it. You do not need to run this notebook or train the model first. Audio is processed in your browser. Allow microphone access to record, or upload an audio file.

This is an experimental research demo, not a medical diagnosis. Training code and evaluation results remain below.


# Respira — original respiratory dataset only

**Five categories:** Asthma, Healthy, Bronchial, COPD and Pneumonia. This workflow uses the original Kaggle respiratory dataset only: 1,211 files, 1,199 after removing exact PCM duplicates. **No ESC-50 recordings are downloaded, loaded, or used for training/calibration.**

There are no noise-labeled examples in this dataset. We cannot create a validated background-noise classifier from disease labels. Instead, the app separately checks silence/clipping, whether audio resembles the training distribution, and whether a prediction meets conservative validation-selected thresholds. Quiet nonzero signals are normalized and analyzed. Unfamiliar or uncertain audio shows exploratory scores without an accepted category label; these previews are not validated diagnoses or confirmed noise.

A 95% validation precision target requires at least 10 accepted validation predictions per class. A category without that evidence is disabled for confident display. This target is not a precision guarantee. Always inspect **test precision, recall and coverage together**. There is no guarantee of zero overfitting, underfitting, or low loss on new phone recordings.

The original exploratory notebook is preserved below, disabled by default. Historical assistant workflows are superseded by this dataset-only workflow.

In [1]:
RUN_LEGACY_NOTEBOOK = False
%pip install numpy scipy scikit-learn soundfile torch onnx onnxruntime pandas matplotlib
from pathlib import Path
import os, json, sys, subprocess, zipfile, urllib.request
ROOT=Path('/content/respira')
ROOT.mkdir(exist_ok=True)
(ROOT/'ml').mkdir(exist_ok=True)
(ROOT/'data').mkdir(exist_ok=True)
os.chdir(ROOT)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 13.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 37.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnx]


## 1. Reuse the original audio files, or restore the same dataset
Colab runtime files are temporary. This first searches existing files; no extra audio dataset is used.

In [2]:
required = {'bronchial', 'asthma', 'copd', 'healthy', 'pneumonia'}
candidates = [Path('/content/Asthma_Detection_Dataset'), ROOT/'data'/'asthma-v2']
# Include an existing dataset_path from an earlier cell in this runtime.
if 'dataset_path' in globals():
    candidates.insert(0, Path(dataset_path))
RESPIRATORY_ROOT = None
for candidate in candidates:
    if candidate.exists():
        folder_labels = {p.parent.name.lower() for p in candidate.rglob('*.wav')}
        if required <= folder_labels:
            RESPIRATORY_ROOT = candidate.resolve()
            break
if RESPIRATORY_ROOT is None:
    archive = ROOT/'data'/'asthma-v2.zip'
    if not archive.exists():
        urllib.request.urlretrieve('https://www.kaggle.com/api/v1/datasets/download/mohammedtawfikmusaed/asthma-detection-dataset-version-2', archive)
    destination = (ROOT/'data'/'asthma-v2').resolve()
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            target = (destination/item.filename).resolve()
            if destination not in target.parents:
                raise ValueError('Unsafe archive path')
            z.extract(item, destination)
    RESPIRATORY_ROOT = destination
print('Using existing/extracted respiratory audio:', RESPIRATORY_ROOT)
from collections import Counter
counts = Counter(p.parent.name for p in RESPIRATORY_ROOT.rglob('*.wav'))
print('All extracted respiratory files:', sum(counts.values()))
display(__import__('pandas').DataFrame(sorted(counts.items()), columns=['Category', 'Recordings']))


Using existing/extracted respiratory audio: /content/respira/data/asthma-v2
All extracted respiratory files: 1211


,Category,Recordings
0,Bronchial,104
1,asthma,288
2,copd,401
3,healthy,133
4,pneumonia,285


### audio_features.py

In [3]:
%%writefile /content/respira/ml/audio_features.py
"""Shared four-second audio preprocessing; no external datasets or model fitting."""
import math
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
SR=8000;SECONDS=4;N=SR*SECONDS;FFT=512;HOP=256;MELS=32;FRAMES=1+(N-FFT)//HOP
SEED=42

def melbank():
 hz=lambda m:700*(10**(m/2595)-1)
 points=hz(np.linspace(2595*np.log10(1+60/700),2595*np.log10(1+1800/700),MELS+2))
 bins=np.arange(FFT//2+1)*SR/FFT
 return np.array([np.maximum(0,np.minimum((bins-a)/(b-a),(c-bins)/(c-b))) for a,b,c in zip(points,points[1:],points[2:])],dtype=np.float32)
BANK=melbank();WINDOW=np.hanning(FFT+1)[:-1].astype(np.float32)
def read_audio(path):
 x,sr=sf.read(path,dtype='float32',always_2d=True);x=x.mean(1)
 g=math.gcd(sr,SR)
 if sr!=SR:x=resample_poly(x,SR//g,sr//g)
 return x.astype(np.float32)
def crop(x,start=0):
 y=x[start:start+N]
 return np.pad(y,(0,max(0,N-len(y)))).astype(np.float32)
def features(x):
 x=x-np.mean(x);peak=np.max(np.abs(x))
 if peak>1e-8:x=x/peak
 frames=np.lib.stride_tricks.sliding_window_view(x,FFT)[::HOP]
 power=np.abs(np.fft.rfft(frames*WINDOW,axis=1))**2
 spec=np.log(np.maximum(BANK@power.T,1e-8)).astype(np.float32)
 return ((spec-spec.mean())/max(float(spec.std()),1e-4))[None]

def softmax(z):
 z=z-z.max(1,keepdims=True);p=np.exp(z);return p/p.sum(1,keepdims=True)


Writing /content/respira/ml/audio_features.py


### train_respiratory.py

In [4]:
%%writefile /content/respira/ml/train_respiratory.py
"""Dataset-only CNN search, selective predictions and transparent held-out evaluation.
No ESC-50 files are read. Noise is not a learned class. All model/threshold choices
use validation data; the reused internal test split is evaluated after selection.
"""
import argparse,copy,hashlib,json,math,random,re,time
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader,TensorDataset,WeightedRandomSampler
from scipy.optimize import minimize_scalar
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score,balanced_accuracy_score
from audio_features import read_audio,crop,features,softmax,SR,N,FFT,HOP,MELS,FRAMES,BANK
LABELS=['asthma','healthy','bronchial','copd','pneumonia']
class CNN(nn.Module):
 def __init__(self,width=2,dropout=.25):
  super().__init__();layers=[];channels=1
  for out in [8*width,16*width,32*width]:
   layers.extend([nn.Conv2d(channels,out,3,padding=1),nn.BatchNorm2d(out),nn.ReLU(),nn.MaxPool2d(2)]);channels=out
  self.conv=nn.Sequential(*layers);self.head=nn.Sequential(nn.Linear(256*width,64*width),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64*width,len(LABELS)))
 def forward(self,x):
  z=self.conv(x);return self.head(torch.cat([z.mean(3),z.amax(3)],dim=1).flatten(1))
class Ensemble(nn.Module):
 def __init__(self,models):super().__init__();self.models=nn.ModuleList(models)
 def forward(self,x):return torch.stack([m(x) for m in self.models]).mean(0)
def metrics(y,p):
 pred=p.argmax(1)
 return {'accuracy':float(accuracy_score(y,pred)),'balanced_accuracy':float(balanced_accuracy_score(y,pred)),'macro_f1':float(f1_score(y,pred,average='macro')),'confusion_matrix':confusion_matrix(y,pred,labels=range(5)).tolist(),'report':classification_report(y,pred,labels=range(5),target_names=LABELS,output_dict=True,zero_division=0)}
def predict(model,x):
 model.eval()
 with torch.no_grad():return np.concatenate([model(torch.from_numpy(chunk)).numpy() for chunk in np.array_split(x,max(1,math.ceil(len(x)/64)))])
def nll(y,p):return float(-np.log(p[np.arange(len(y)),y]+1e-9).mean())
def descriptor(x):
 z=x[:,0];return np.concatenate([z.mean(2),z.std(2)],axis=1)
def distances(x,gate):
 d=descriptor(x);return np.sqrt(np.min(np.mean(((d[:,None,:]-np.array(gate['centroids'])[None,:,:])/np.array(gate['scale']))**2,axis=2),axis=1))
def selective(y,p,accepted):
 pred=p.argmax(1);by={}
 for c,name in enumerate(LABELS):
  selected=accepted&(pred==c);tp=int(((y==c)&selected).sum());count=int(selected.sum())
  by[name]={'precision':tp/count if count else None,'accepted_predictions':count,'true_positives':tp,'recall_after_abstention':tp/int((y==c).sum()) if (y==c).any() else None}
 return {'accuracy':float((pred[accepted]==y[accepted]).mean()) if accepted.any() else None,'coverage':float(accepted.mean()),'accepted':int(accepted.sum()),'total':len(y),'per_class':by}
def main():
 ap=argparse.ArgumentParser();ap.add_argument('--data',type=Path,required=True);ap.add_argument('--out',type=Path,required=True);ap.add_argument('--respiratory-root',type=Path);args=ap.parse_args()
 torch.set_num_threads(4);torch.manual_seed(42);random.seed(42);np.random.seed(42);rng=np.random.default_rng(42)
 out=args.out;out.mkdir(parents=True,exist_ok=True);rows=[];waves=[];seen={};duplicates=0
 root=args.respiratory_root or args.data/'asthma-v2'
 for path in sorted(root.rglob('*.wav')):
  name=path.parent.name.lower()
  if name not in LABELS:continue
  y=LABELS.index(name);x=read_audio(path);digest=hashlib.sha256(x.tobytes()).hexdigest()
  if digest in seen:
   if seen[digest]!=y:raise ValueError('Conflicting duplicate labels')
   duplicates+=1;continue
  seen[digest]=y;rows.append({'path':str(path.resolve()),'source':'asthma-v2','label':y,'group':'resp-'+re.match(r'P\d+',path.name).group(),'seconds':len(x)/SR,'sha256_pcm':digest});waves.append(x)
 y=np.array([r['label'] for r in rows]);groups=np.array([r['group'] for r in rows]);split=np.full(len(rows),'train',dtype=object)
 # Keep the exact previous respiratory split: broad-label stratification is retained.
 folds=list(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42).split(np.arange(len(y)),np.where(y>=2,3,y),groups));split[folds[0][1]]='test';split[folds[1][1]]='validation'
 for a,b in [('train','test'),('train','validation'),('validation','test')]:assert not set(groups[split==a])&set(groups[split==b])
 for i,r in enumerate(rows):r['split']=split[i]
 (out/'manifest.json').write_text(json.dumps(rows,indent=2));print('Dataset only:',len(rows),'unique files; split:',{s:int((split==s).sum()) for s in ['train','validation','test']},flush=True)
 first=np.stack([features(crop(x)) for x in waves]);last=np.stack([features(crop(x,max(0,len(x)-N))) for x in waves]);ids=np.where(split=='train')[0];vi=np.where(split=='validation')[0];ti=np.where(split=='test')[0]
 # Augmentation and all duplicates/crops stay inside original recording groups.
 crops=np.stack([features(crop(waves[i],int(rng.integers(0,max(1,len(waves[i])-N+1))))) for i in ids for _ in range(2)])
 candidates=[{'name':'wide','width':2,'dropout':.25,'lr':.001,'random_crops':False,'seed':42},{'name':'wide-crops','width':2,'dropout':.35,'lr':.0007,'random_crops':True,'seed':43},{'name':'compact','width':1,'dropout':.25,'lr':.001,'random_crops':True,'seed':44}]
 models=[];records=[];vzs=[]
 for spec in candidates:
  torch.manual_seed(spec['seed']);rng=np.random.default_rng(spec['seed']);x=np.concatenate([first[ids],last[ids]]+([crops] if spec['random_crops'] else []));ty=np.concatenate([y[ids],y[ids]]+([np.repeat(y[ids],2)] if spec['random_crops'] else []));weights=1/np.bincount(ty,minlength=5)
  sampler=WeightedRandomSampler(torch.tensor(weights[ty]),len(ty),replacement=True);loader=DataLoader(TensorDataset(torch.from_numpy(x),torch.from_numpy(ty).long()),batch_size=48,sampler=sampler)
  model=CNN(spec['width'],spec['dropout']);optimizer=torch.optim.AdamW(model.parameters(),lr=spec['lr'],weight_decay=.005);scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,80);best=-1;best_epoch=0;history=[];started=time.time()
  for epoch in range(80):
   model.train();total=0
   for xb,yb in loader:
    if rng.random()<.6:
     f=int(rng.integers(MELS-2));xb[:,:,f:f+2,:]=0
    if rng.random()<.6:
     t=int(rng.integers(FRAMES-8));xb[:,:,:,t:t+8]=0
    optimizer.zero_grad();logits=model(xb);loss=nn.functional.cross_entropy(logits,yb,label_smoothing=.03);loss.backward();nn.utils.clip_grad_norm_(model.parameters(),5);optimizer.step();total+=loss.item()*len(yb)
   scheduler.step();vz=predict(model,first[vi]);p=softmax(vz);m=metrics(y[vi],p);score=m['macro_f1'];history.append({'epoch':epoch+1,'training_augmented_loss':total/len(ty),'validation_loss':nll(y[vi],p),'validation_accuracy':m['accuracy'],'validation_macro_f1':score})
   print(spec['name'],'epoch',epoch+1,'loss',round(total/len(ty),3),'val_loss',round(history[-1]['validation_loss'],3),'val_acc',round(m['accuracy'],3),'val_f1',round(score,3),flush=True)
   if score>best:best=score;best_epoch=epoch+1;best_state=copy.deepcopy(model.state_dict())
   if epoch+1-best_epoch>=16 and epoch>=35:break
  model.load_state_dict(best_state);model.eval();torch.save(best_state,out/(spec['name']+'.pt'));vz=predict(model,first[vi]);models.append(model);vzs.append(vz)
  records.append({'name':spec['name'],'parameters':spec,'best_epoch':best_epoch,'epochs_run':len(history),'training_seconds':time.time()-started,'history':history,'validation':metrics(y[vi],softmax(vz))})
 # Compare fixed, equal-weight ensembles on validation before touching test inference.
 choices=[([i],vzs[i]) for i in range(len(models))]+[([0,1],(vzs[0]+vzs[1])/2),([0,1,2],np.mean(vzs,axis=0))]
 scored=[{'members':ix,'validation':metrics(y[vi],softmax(z))} for ix,z in choices];win=max(range(len(scored)),key=lambda i:scored[i]['validation']['macro_f1']);chosen=scored[win]['members'];model=Ensemble([models[i] for i in chosen]).eval();vz=choices[win][1]
 selection={'criterion':'Validation first-window macro F1, then fixed equal-weight ensemble comparison','candidates':records,'choices':scored,'selected_members':[candidates[i]['name'] for i in chosen],'test_used_for_selection':False};(out/'model_selection.json').write_text(json.dumps(selection,indent=2));print('SELECTION FROZEN',selection['selected_members'],flush=True)
 temp=float(minimize_scalar(lambda t:nll(y[vi],softmax(vz/t)),bounds=(.5,5),method='bounded').x);vp=softmax(vz/temp)
 # A one-class input support check uses only dataset recordings. It is NOT a noise classifier.
 d=descriptor(first[ids]);scale=np.maximum(d.std(0),.1);centroids=[d[y[ids]==c].mean(0).tolist() for c in range(5)];gate={'centroids':centroids,'scale':scale.tolist()};vd=distances(first[vi],gate);gate['maxDistance']=float(np.quantile(vd,.99));gate['note']='Dataset-support heuristic, not a validated noise detector; fitted on train and validation respiratory recordings only.'
 margin=.15;vsort=np.sort(vp,axis=1);eligible=(vd<=gate['maxDistance'])&(vsort[:,-1]-vsort[:,-2]>=margin);thresholds=[];threshold_details=[]
 for cls in range(5):
  good=[]
  for threshold in np.arange(.5,1.0,.025):
   accepted=eligible&(vp.argmax(1)==cls)&(vp[:,cls]>=threshold);count=int(accepted.sum());precision=float((y[vi][accepted]==cls).mean()) if count else 0
   if count>=10 and precision>=.95:good.append((float(threshold),count,precision))
  threshold,count,precision=min(good) if good else (1.01,0,None);thresholds.append(threshold);threshold_details.append({'label':LABELS[cls],'threshold':threshold,'validation_accepted':count,'validation_precision':precision,'enabled':bool(good)})
 def accept(x,p):
  sp=np.sort(p,axis=1);return (distances(x,gate)<=gate['maxDistance'])&(sp[:,-1]-sp[:,-2]>=margin)&(p.max(1)>=np.array(thresholds)[p.argmax(1)])
 # Test inference starts only after model, calibration, gates and thresholds are frozen.
 zfirst=predict(model,first[ti]);zlast=predict(model,last[ti]);tp=softmax((zfirst+zlast)/2/temp);fp=softmax(zfirst/temp);complete=np.array([len(waves[i])>=N for i in ti]);fids=ti[complete];fp=fp[complete];eligible_first=accept(first[fids],fp);trainp=softmax(predict(model,first[ids])/temp)
 allx=[];ally=[];windowids=[]
 for i in ti:
  for end in range(N,len(waves[i])+1,SR):allx.append(features(waves[i][end-N:end]));ally.append(y[i]);windowids.append(int(i))
 allx=np.stack(allx);ally=np.array(ally);wp=softmax(predict(model,allx)/temp);wa=accept(allx,wp)
 report={'version':'respira-cnn-v4-dataset-only','labels':LABELS,'recordings':len(rows),'external_training_recordings':0,'respiratory_exact_duplicates_removed':duplicates,'split_counts':{s:np.bincount(y[split==s],minlength=5).tolist() for s in ['train','validation','test']},'split_policy':'Unchanged global P-number proxy groups; exact PCM duplicates removed. P-number identity is not independently verified. Existing test set has been inspected in earlier development, not external validation.','selected_members':selection['selected_members'],'model_selection':selection,'validation':metrics(y[vi],vp),'test':metrics(y[ti],tp),'first_window_evaluation':{'seconds':4,'recordings':len(fids),'test':metrics(y[fids],fp),'selective':selective(y[fids],fp,eligible_first)},'window_evaluation':{'window_seconds':4,'hop_seconds':1,'test_windows':len(ally),'test_recordings':len(set(windowids)),'test':metrics(ally,wp),'selective':selective(ally,wp,wa),'note':'Overlapping windows are correlated, not independent participants.'},'selective_policy':{'target_validation_precision':.95,'minimum_validation_predictions_per_class':10,'threshold_details':threshold_details,'margin':margin,'validation':selective(y[vi],vp,accept(first[vi],vp)),'note':'95% is a validation selection target, not a guaranteed precision. Unsupported classes remain disabled. Coverage and per-class recall include the cost of abstention.'},'fit_diagnostics':{'train_first_window_loss':nll(y[ids],trainp),'validation_first_window_loss':nll(y[vi],vp),'train_first_window_accuracy':metrics(y[ids],trainp)['accuracy'],'validation_first_window_accuracy':metrics(y[vi],vp)['accuracy'],'note':'Comparable unaugmented first-window losses after temperature calibration. Curves and train/validation gaps are evidence, not a guarantee of no overfitting or underfitting.'},'temperature':temp,'thresholds':thresholds,'limitations':['No noise-labeled examples: there is no learned background-noise class.','Dataset support rejection cannot reliably distinguish noise from unfamiliar respiratory recordings.','Phone microphone and cough diagnosis are unvalidated; source recordings include stethoscope lung sounds.','Clinical precision is unknown. Internal thresholds may not generalize to real-world audio.','Healthy is a dataset label and cannot confirm health.']}
 report['accuracy_target']={'target':.90,'achieved':bool(report['test']['accuracy']>=.90 and report['first_window_evaluation']['test']['accuracy']>=.90)}
 dummy=torch.from_numpy(first[:1]);torch.onnx.export(model,dummy,str(out/'respira.onnx'),input_names=['features'],output_names=['logits'],opset_version=17,dynamo=False)
 import onnxruntime as ort
 options=ort.SessionOptions();options.intra_op_num_threads=1;session=ort.InferenceSession(str(out/'respira.onnx'),sess_options=options,providers=['CPUExecutionProvider']);actual=session.run(None,{'features':first[:1]})[0];report['onnx_max_abs_error']=float(np.max(np.abs(actual-model(dummy).detach().numpy())))
 times=[]
 for _ in range(30):
  start=time.perf_counter();session.run(None,{'features':first[:1]});times.append((time.perf_counter()-start)*1000)
 report['local_onnx_p50_ms']=float(np.median(times))
 config={'version':report['version'],'labels':LABELS,'sampleRate':SR,'windowSeconds':4,'hopSeconds':1,'fftSize':FFT,'hopSize':HOP,'melBands':MELS,'frames':FRAMES,'melBank':BANK.tolist(),'temperature':temp,'thresholds':thresholds,'minimumMargin':margin,'supportGate':gate,'silenceRms':.0005,'modelFile':'respira.onnx','limitations':report['limitations']}
 (out/'config.json').write_text(json.dumps(config));(out/'metrics.json').write_text(json.dumps(report,indent=2));np.savez_compressed(out/'parity_fixture.npz',wave=crop(waves[0]),features=first[0]);(out/'test_predictions.json').write_text(json.dumps([{'file':Path(rows[i]['path']).name,'truth':LABELS[y[i]],'scores':p.tolist()} for i,p in zip(ti,tp)],indent=2));print('FINAL',json.dumps({'recording_accuracy':report['test']['accuracy'],'first_accuracy':report['first_window_evaluation']['test']['accuracy'],'selective':report['first_window_evaluation']['selective'],'fit':report['fit_diagnostics']}),flush=True)
if __name__=='__main__':main()


Writing /content/respira/ml/train_respiratory.py


### audit_evaluation.py

In [5]:
%%writefile /content/respira/ml/audit_evaluation.py
"""Audit a frozen model using the app's full first-window decision rules.
No fitting, checkpoint selection, threshold changes, or external audio occurs.
Bootstrap intervals resample filename groups, not verified patient identities.
Usage: python ml/audit_evaluation.py MODEL_OUTPUT
"""
import csv,hashlib,json,sys
from pathlib import Path
import numpy as np
import onnxruntime as ort
from audio_features import read_audio,crop,features,softmax,N

def audit(out):
 out=Path(out);cfg=json.loads((out/'config.json').read_text());report=json.loads((out/'metrics.json').read_text());rows=json.loads((out/'manifest.json').read_text());labels=cfg['labels'];k=len(labels)
 policy_path=out/'audio-policy.json'
 policy=json.loads(policy_path.read_text()) if policy_path.exists() else {'version': 'mic-preview-v1', 'noSignalRms': 1e-07, 'noSignalPeak': 1e-06, 'quietWarningRms': 0.0005, 'clippedFraction': 0.08, 'unfamiliarAudio': 'preview_only', 'uncertainAudio': 'preview_only', 'captureAutoGainControl': True, 'note': 'Engineering input policy, not a clinically calibrated noise detector. Quiet nonzero signals are normalized by existing feature preprocessing. Preview scores do not bypass the original category acceptance thresholds.'}
 policy_path.write_text(json.dumps(policy,indent=2))
 options=ort.SessionOptions();options.intra_op_num_threads=1;session=ort.InferenceSession(str(out/'respira.onnx'),sess_options=options,providers=['CPUExecutionProvider'])
 results=[];probabilities=[];excluded=[]
 for r in rows:
  if r['split']!='test':continue
  wave=read_audio(r['path'])
  if len(wave)<N:excluded.append({'recording':Path(r['path']).name,'reason':'shorter than four seconds','label':labels[r['label']]});continue
  x=crop(wave);f=features(x);logits=session.run(None,{'features':f[None]})[0];p=softmax(logits/cfg['temperature'])[0];pred=int(p.argmax());ordered=np.sort(p)
  means=f[0].mean(1);sd=f[0].std(1);d=np.concatenate([means,sd]);gate=cfg['supportGate'];distance=float(np.sqrt(np.min(np.mean(((d-np.array(gate['centroids']))/np.array(gate['scale']))**2,axis=1))))
  ac=x.astype(np.float64)-x.astype(np.float64).mean();rms=float(np.sqrt(np.mean(ac*ac)));peak=float(np.abs(ac).max());clipped=float((np.abs(x)>.99).mean())
  reason='no_signal' if rms<policy['noSignalRms'] or peak<policy['noSignalPeak'] else 'clipping' if clipped>policy['clippedFraction'] else 'unsupported_audio' if distance>gate['maxDistance'] else 'below_class_threshold' if p[pred]<cfg['thresholds'][pred] else 'ambiguous_scores' if ordered[-1]-ordered[-2]<cfg['minimumMargin'] else 'accepted'
  results.append({'recording':Path(r['path']).name,'group':r['group'],'truth':labels[r['label']],'predicted':labels[pred],'shown_in_app':labels[pred] if reason=='accepted' else 'abstain','reason':reason,'exploratory_preview':reason not in ['accepted','no_signal','clipping'],'quiet_signal':rms<policy['quietWarningRms'],'top_score':float(p[pred]),'rms':rms,'clipped_fraction':clipped,'support_distance':distance});probabilities.append(p)
 y=np.array([labels.index(r['truth']) for r in results]);pred=np.array([labels.index(r['predicted']) for r in results]);take=np.array([r['reason']=='accepted' for r in results]);groups=np.array([r['group'] for r in results]);p=np.stack(probabilities)
 raw=np.bincount(y*k+pred,minlength=k*k).reshape(k,k);shown=np.bincount(y*(k+1)+np.where(take,pred,k),minlength=k*(k+1)).reshape(k,k+1)
 def ratio(a,b):return float(a/b) if b else None
 def summarize(raw,shown):
  total=int(raw.sum());accepted=int(shown[:,:k].sum());correct=int(np.trace(raw));accepted_correct=int(np.trace(shown[:,:k]));classes=[]
  for c,label in enumerate(labels):
   count=int(raw[c].sum());npred=int(raw[:,c].sum());nshow=int(shown[:,c].sum());tp=int(shown[c,c])
   classes.append({'label':label,'support':count,'raw_true_positives':int(raw[c,c]),'raw_predictions':npred,'raw_precision':ratio(raw[c,c],npred),'raw_recall':ratio(raw[c,c],count),'accepted_predictions':nshow,'accepted_true_positives':tp,'accepted_false_positives':nshow-tp,'accepted_precision':ratio(tp,nshow),'recall_after_abstention':ratio(tp,count),'abstentions_for_true_class':int(shown[c,-1])})
  return {'recordings':total,'correct':correct,'accuracy':ratio(correct,total),'accepted':accepted,'accepted_correct':accepted_correct,'accepted_errors':accepted-accepted_correct,'accepted_precision':ratio(accepted_correct,accepted),'coverage':ratio(accepted,total),'abstained':total-accepted,'classes':classes}
 summary=summarize(raw,shown);unique=np.unique(groups);raw_groups=[];shown_groups=[]
 for group in unique:
  mask=groups==group;raw_groups.append(np.bincount(y[mask]*k+pred[mask],minlength=k*k).reshape(k,k));shown_groups.append(np.bincount(y[mask]*(k+1)+np.where(take[mask],pred[mask],k),minlength=k*(k+1)).reshape(k,k+1))
 raw_groups=np.stack(raw_groups);shown_groups=np.stack(shown_groups);rng=np.random.default_rng(20261114);boots=[]
 for _ in range(2000):
  ix=rng.integers(0,len(unique),len(unique));boots.append(summarize(raw_groups[ix].sum(0),shown_groups[ix].sum(0)))
 def interval(values):
  valid=[v for v in values if v is not None];return {'low':float(np.percentile(valid,2.5)) if valid else None,'high':float(np.percentile(valid,97.5)) if valid else None,'valid_resamples':len(valid)}
 intervals={metric:interval([b[metric] for b in boots]) for metric in ['accuracy','accepted_precision','coverage']}
 for i,c in enumerate(summary['classes']):c['accepted_precision_interval']=interval([b['classes'][i]['accepted_precision'] for b in boots])
 confidence=p.max(1);correct=(pred==y);bins=[];ece=0
 for i in range(10):
  mask=(confidence>=i/10)&(confidence<(i+1)/10 if i<9 else confidence<=1);count=int(mask.sum());avg=float(confidence[mask].mean()) if count else None;acc=float(correct[mask].mean()) if count else None
  if count:ece+=count/len(y)*abs(avg-acc)
  bins.append({'lower':i/10,'upper':(i+1)/10,'count':count,'mean_score':avg,'accuracy':acc})
 baseline=int(np.bincount([r['label'] for r in rows if r['split']=='train'],minlength=k).argmax())
 comparison={'old_report_first_window_accuracy':report['first_window_evaluation']['test']['accuracy'],'old_report_accepted':report['first_window_evaluation']['selective']['accepted'],'current_app_accepted':summary['accepted'],'note':'Current audit includes silence and clipping checks as well as support distance, margin and class thresholds. Original model selection report did not include the two signal-quality checks.'}
 assert abs(summary['accuracy']-comparison['old_report_first_window_accuracy'])<1e-10
 result={'schema_version':1,'model_version':cfg['version'],'audit_date':'2026-11-14','status':'Internal research benchmark; not clinical validation','protocol':{'implementation':'Python/ONNX audit of the browser decision rules; soundfile/SciPy decoding and resampling. This is not an end-to-end trial across browsers or microphones.','unit':'One first complete four-second window per eligible test recording','recordings':len(results),'excluded':excluded,'split_counts':{s:sum(r['split']==s for r in rows) for s in ['train','validation','test']},'group_count':len(unique),'groups':unique.tolist(),'patient_identity_verified':False,'test_previously_inspected':True,'external_audio_used':False,'model_or_class_thresholds_changed':False,'input_policy_version':policy['version'],'display_note':'Accepted-label metrics exclude exploratory score previews. Near-digital silence blocks analysis; quiet signals are analyzed. Unfamiliar and uncertain audio may show unvalidated previews without assigning a category.','confidence_interval_method':'Descriptive 95% percentile cluster bootstrap, 2,000 resamples, seed 20261114. Resamples P-number filename groups with replacement. Groups are unverified source/patient proxies; intervals do not establish clinical confidence or independent external validation. Per-class precision intervals omit resamples with zero accepted predictions for that class; valid counts are recorded.'},'summary':summary,'intervals':intervals,'confusion':{'labels':labels,'all_predictions':raw.tolist(),'app_display_columns':labels+['abstain'],'app_display':shown.tolist()},'baseline':{'policy':'Always predict the largest training class','label':labels[baseline],'correct':int((y==baseline).sum()),'accuracy':float((y==baseline).mean())},'calibration':{'ece_10_equal_width_bins':float(ece),'multiclass_brier_score':float(np.mean(np.sum((p-np.eye(k)[y])**2,axis=1))),'negative_log_likelihood':float(-np.mean(np.log(p[np.arange(len(y)),y]+1e-9))),'bins':bins,'note':'Calibration describes dataset-label predictions before abstention. These are not probabilities of disease in a person.'},'abstention_reasons':{reason:sum(r['reason']==reason for r in results) for reason in sorted(set(r['reason'] for r in results)) if reason!='accepted'},'fit_diagnostics':report['fit_diagnostics'],'selected_members':report['selected_members'],'thresholds':report['selective_policy']['threshold_details'],'earlier_report_comparison':comparison,'artifacts':{'audio_policy_sha256':hashlib.sha256(policy_path.read_bytes()).hexdigest(),'model_sha256':hashlib.sha256((out/'respira.onnx').read_bytes()).hexdigest(),'config_sha256':hashlib.sha256((out/'config.json').read_bytes()).hexdigest(),'dataset_split_sha256':hashlib.sha256(json.dumps(sorted((r['sha256_pcm'],r['label'],r['split']) for r in rows),separators=(',',':')).encode()).hexdigest()},'limitations':['No external microphone or clinical validation.','Repeated use of this test benchmark can introduce development bias.','P-number groups may not correspond to unique patients.','Few groups and small per-class counts make intervals unstable.','Abstention can improve displayed precision while missing true conditions.','No labeled noise set: noise-detection performance is unknown.']}
 (out/'evaluation.json').write_text(json.dumps(result,indent=2))
 with (out/'evaluation-predictions.csv').open('w') as f:
  writer=csv.DictWriter(f,fieldnames=list(results[0]));writer.writeheader();writer.writerows(results)
 print(json.dumps({'summary':summary,'group_count':len(unique),'intervals':intervals,'comparison':comparison},indent=2))
 return result
if __name__=='__main__':audit(sys.argv[1])


Writing /content/respira/ml/audit_evaluation.py


## 2. Train and select on validation data

The unchanged filename-group split has 715 training, 265 validation and 219 test recordings. Patient identity is not independently verified; the test benchmark was inspected during earlier work and is not an external clinical evaluation.

Three CNN configurations use balanced sampling, dropout, weight decay, training-only spectrogram masks, optional random crops, gradient clipping, learning-rate decay, and early stopping. Validation first-window macro F1 chooses checkpoints and compares fixed equal-weight ensembles. Model choice, temperature calibration, input-support distance and class thresholds are frozen **before test inference**. No noise clips or extracted “noise segments” are fabricated from respiratory recordings.

In [ ]:
subprocess.run([sys.executable,'-u','ml/train_respiratory.py',
                '--data','data','--respiratory-root',str(RESPIRATORY_ROOT),
                '--out','training-respiratory-output'],check=True)


## Audit the frozen model with the app display rules
This adds signal-quality checks, group-bootstrap ranges, calibration diagnostics and prediction-level exports. It does not retrain or change thresholds. Python audio decoding/resampling is not an end-to-end browser or clinical test.

In [ ]:
subprocess.run([sys.executable,'ml/audit_evaluation.py','training-respiratory-output'],check=True)


## 3. Inspect accuracy, precision, recall, abstention and loss
The first-window report matches early prediction. Recording accuracy averages first/last windows. Overlapping streaming windows are correlated and must not be counted as independent people.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
out=Path('training-respiratory-output')
r=json.loads((out/'metrics.json').read_text())
print('Selected:',r['selected_members'])
print('All-recording accuracy:',r['test']['accuracy'])
print('First-four-second accuracy:',r['first_window_evaluation']['test']['accuracy'])
display(pd.DataFrame(r['first_window_evaluation']['test']['report']).T)
audit=json.loads((out/'evaluation.json').read_text())
selected=audit['summary']
print('Precision among accepted first-window category labels:',selected['accepted_precision'])
print('Coverage:',selected['coverage'],'Accepted:',selected['accepted'],'of',selected['recordings'])
display(pd.DataFrame(selected['classes']).set_index('label'))
display(pd.DataFrame(r['selective_policy']['threshold_details']))
print('Train/validation comparison:',r['fit_diagnostics'])
fig,axes=plt.subplots(len(r['model_selection']['candidates']),2,figsize=(12,10))
for row,candidate in enumerate(r['model_selection']['candidates']):
    h=pd.DataFrame(candidate['history'])
    axes[row,0].plot(h.epoch,h.training_augmented_loss,label='Augmented training loss')
    axes[row,0].plot(h.epoch,h.validation_loss,label='Unaugmented validation loss')
    axes[row,0].set(title=candidate['name'],xlabel='Epoch',ylabel='Loss')
    axes[row,0].legend()
    axes[row,1].plot(h.epoch,h.validation_accuracy,label='Validation accuracy')
    axes[row,1].plot(h.epoch,h.validation_macro_f1,label='Validation macro F1')
    axes[row,1].axvline(candidate['best_epoch'],linestyle='--',color='gray')
    axes[row,1].set(xlabel='Epoch',ylim=(0,1))
    axes[row,1].legend()
plt.tight_layout();plt.show()
print('Training curve uses augmentation and label smoothing, so its loss is not directly comparable to validation loss. Use the unaugmented fit_diagnostics comparison above.')
cm=np.array(r['first_window_evaluation']['test']['confusion_matrix'])
fig,ax=plt.subplots(figsize=(8,6));im=ax.imshow(cm,cmap='Blues')
ax.set(xticks=range(5),yticks=range(5),xticklabels=r['labels'],yticklabels=r['labels'],xlabel='Predicted',ylabel='Actual',title='First four seconds — all predictions, including abstentions')
for i in range(5):
    for j in range(5):ax.text(j,i,str(cm[i,j]),ha='center',va='center')
fig.colorbar(im,ax=ax);plt.tight_layout();plt.show()


## 4. Export matched browser assets
Keep weights and configuration together. Five outputs cannot be substituted into an old six-output configuration. The public app runs these assets locally; changing the notebook does not automatically redeploy the website.

In [ ]:
with zipfile.ZipFile('respira-dataset-only-model.zip','w',zipfile.ZIP_DEFLATED) as z:
    for name in ['respira.onnx','config.json','metrics.json','evaluation.json','evaluation-predictions.csv','audio-policy.json']:
        z.write(out/name,arcname=name)
print('Export:',ROOT/'respira-dataset-only-model.zip')
# from google.colab import files
# files.download('respira-dataset-only-model.zip')


## Latest completed local run — reference results

These results come from actual local training on November 14, 2026. The assistant did not execute these new cells inside your Colab runtime.

- Selected: wide-crops.
- All-recording accuracy: 79.5%.
- First-four-second accuracy: 78.0%.
- Precision among accepted first-window predictions: 88.1%, at 57.8% coverage (126 of 218 recordings).
- 90% all-recording/first-window accuracy target: not met.

The full-rule audit includes clipping and silence checks omitted from the earlier selective training report. This reduces accepted first windows from 130 to 126. Descriptive group-bootstrap ranges and per-recording decisions are in the exported evaluation files. Model weights and category acceptance thresholds are unchanged. The mic-preview-v1 engineering input policy accepts quiet nonzero signals and exposes clearly labeled exploratory previews; it is not calibration on real microphone recordings.

These metrics do not establish clinical accuracy or robustness to phone-microphone noise. No noise-detection accuracy can be measured from a dataset without noise labels.


## Original exploratory notebook — preserved, disabled by default
Keep `RUN_LEGACY_NOTEBOOK=False` to avoid running the historical pipeline and its old dependencies. Its outputs are historical, not evidence for the current model.

In [ ]:
if RUN_LEGACY_NOTEBOOK:
    #imports
    import os
    import librosa
    import librosa.display
    import numpy as np
    import matplotlib.pyplot as plt
    from prettytable import PrettyTable
    import seaborn as sns
    import pandas as pd


    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder, label_binarize
    from sklearn.decomposition import PCA

    from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import (
        Conv1D, MaxPooling1D, Flatten, LSTM, Dense, Dropout,
        Bidirectional, Attention, Input, LayerNormalization
    )
    import random
    # If running locally or on some platforms, uncomment to install shap:

    import shap


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    #The dataset path
    dataset_path = '/content/Asthma_Detection_Dataset'

    # The five categories/folders we have in the dataset
    categories = ['Bronchial', 'pneumonia', 'asthma', 'healthy', 'copd']

    # Print out the list of categories
    print("Categories:", categories)


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 3) BAR CHART: NUMBER OF AUDIO FILES IN EACH CATEGORY
    ##############################################################################
    # Count how many .wav files exist in each category folder
    category_counts = {
        category: len([
            f for f in os.listdir(os.path.join(dataset_path, category))
            if f.endswith('.wav')
        ])
        for category in categories
    }

    # Plot a bar chart of category counts
    plt.figure(figsize=(8, 6))
    plt.bar(category_counts.keys(), category_counts.values())
    plt.xlabel('Category')
    plt.ylabel('Number of Files')
    plt.title('Number of Audio Files in Each Category')
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 4) TABLE: NUMBER OF AUDIO FILES IN EACH CATEGORY
    ##############################################################################
    # Create a PrettyTable to show the category counts
    table = PrettyTable()
    table.field_names = ["Category", "Number of Audio Files"]
    for category, count in category_counts.items():
        table.add_row([category, count])

    print("Number of Audio Files in Each Category:")
    print(table)


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 5) SHOW 5 WAVEFORM PLOTS (ONE FROM EACH CATEGORY)
    ##############################################################################
    # The user wants exactly 5 waveforms, one for each category.
    # We will loop through the 5 categories, load the first audio in each folder,
    # and plot the waveform using librosa.display.waveshow.
    print("\n=== Waveform Examples for Each Category ===")
    for i, category in enumerate(categories):
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        if i < 5:  # just making sure we match exactly 5 categories
            if audio_files:
                audio_path = os.path.join(category_path, audio_files[0])
                y, sr = librosa.load(audio_path, sr=None)

                plt.figure(figsize=(10, 4))
                librosa.display.waveshow(y, sr=sr)
                plt.title(f'Waveform - {category}')
                plt.xlabel('Time (s)')
                plt.ylabel('Amplitude')
                plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 6) SHOW 5 MEL SPECTROGRAMS (ONE FROM EACH CATEGORY)
    ##############################################################################
    # The user wants a standard spectrogram.
    # Here, we use librosa.feature.melspectrogram -> power_to_db -> specshow.
    print("\n=== Mel Spectrogram Examples for Each Category ===")
    for i, category in enumerate(categories):
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        if i < 5:
            if audio_files:
                audio_path = os.path.join(category_path, audio_files[0])
                y, sr = librosa.load(audio_path, sr=None)

                mel_spect = librosa.feature.melspectrogram(y=y, sr=sr)
                mel_spect_db = librosa.power_to_db(mel_spect, ref=np.max)

                plt.figure(figsize=(10, 4))
                librosa.display.specshow(mel_spect_db, sr=sr, x_axis='time', y_axis='mel')
                plt.colorbar(format='%+2.0f dB')
                plt.title(f'Mel Spectrogram - {category}')
                plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 7) SHOW 5 MFCC PLOTS (ONE FROM EACH CATEGORY)
    ##############################################################################
    # We display the 13 MFCC coefficients with librosa.display.specshow.
    print("\n=== MFCC Examples for Each Category ===")
    for i, category in enumerate(categories):
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        if i < 5:
            if audio_files:
                audio_path = os.path.join(category_path, audio_files[0])
                y, sr = librosa.load(audio_path, sr=None)

                mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
                plt.figure(figsize=(10, 4))
                librosa.display.specshow(mfccs, sr=sr, x_axis='time')
                plt.colorbar()
                plt.title(f'MFCCs - {category}')
                plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 8) SHOW 5 LOG POWER SPECTROGRAMS (ONE FROM EACH CATEGORY)
    ##############################################################################
    # Using STFT -> power_to_db -> specshow with y_axis='log'
    print("\n=== Log Power Spectrogram Examples for Each Category ===")
    for i, category in enumerate(categories):
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        if i < 5:
            if audio_files:
                audio_path = os.path.join(category_path, audio_files[0])
                y, sr = librosa.load(audio_path, sr=None)

                D = np.abs(librosa.stft(y))**2
                log_power_spectrogram = librosa.power_to_db(D, ref=np.max)

                plt.figure(figsize=(10, 4))
                librosa.display.specshow(
                    log_power_spectrogram, sr=sr, x_axis='time', y_axis='log'
                )
                plt.colorbar(format='%+2.0f dB')
                plt.title(f'Log Power Spectrogram - {category}')
                plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 9) SHOW 5 ZERO CROSSING RATE PLOTS (ONE FROM EACH CATEGORY)
    ##############################################################################
    # We'll compute zero_crossing_rate and simply plot the array.
    print("\n=== Zero Crossing Rate Examples for Each Category ===")
    for i, category in enumerate(categories):
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        if i < 5:
            if audio_files:
                audio_path = os.path.join(category_path, audio_files[0])
                y, sr = librosa.load(audio_path, sr=None)

                zero_crossings = librosa.feature.zero_crossing_rate(y)[0]

                plt.figure(figsize=(10, 4))
                plt.plot(zero_crossings, label='Zero Crossing Rate')
                plt.xlabel('Frames')
                plt.ylabel('Rate')
                plt.title(f'Zero-Crossing Rate - {category}')
                plt.legend()
                plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 10) SHOW 5 CHROMA FEATURE PLOTS (ONE FROM EACH CATEGORY)
    ##############################################################################
    # We'll compute the chroma from STFT and plot it.
    print("\n=== Chroma Feature Examples for Each Category ===")
    for i, category in enumerate(categories):
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        if i < 5:
            if audio_files:
                audio_path = os.path.join(category_path, audio_files[0])
                y, sr = librosa.load(audio_path, sr=None)

                chroma = librosa.feature.chroma_stft(y=y, sr=sr)
                plt.figure(figsize=(10, 4))
                librosa.display.specshow(chroma, sr=sr, x_axis='time', y_axis='chroma')
                plt.colorbar()
                plt.title(f'Chroma Features - {category}')
                plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 11) DISTRIBUTION OF AUDIO DURATIONS BY CATEGORY
    ##############################################################################
    # We create a boxplot of the duration distribution for each category.
    durations = []
    labels = []

    print("\n=== Distribution of Audio Durations by Category ===")
    for category in categories:
        category_path = os.path.join(dataset_path, category)
        for audio_file in os.listdir(category_path):
            if audio_file.endswith('.wav'):
                audio_path = os.path.join(category_path, audio_file)
                y, sr = librosa.load(audio_path, sr=None)
                durations.append(librosa.get_duration(y=y, sr=sr))
                labels.append(category)

    df = pd.DataFrame({'Duration': durations, 'Category': labels})
    plt.figure(figsize=(8, 6))
    sns.boxplot(x='Category', y='Duration', data=df)
    plt.title('Distribution of Audio Durations by Category')
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 12) MFCC HEATMAP (FOR A SINGLE AUDIO EXAMPLE)
    ##############################################################################
    # We'll just use the last loaded audio sample (y, sr) from above
    # or load one if you prefer a guaranteed file. Let's do it with the last file
    # that we have in memory. If needed, pick any sample again.
    mfcc_sample = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    plt.figure(figsize=(10, 6))
    sns.heatmap(mfcc_sample, cmap='coolwarm')
    plt.title('MFCC Heatmap')
    plt.show()



In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 13) AVERAGE MFCCs ACROSS CATEGORIES
    ##############################################################################
    # We'll compute average MFCC values across all files in each category.
    avg_mfccs = []

    for category in categories:
        category_path = os.path.join(dataset_path, category)
        mfccs_for_cat = []
        for audio_file in os.listdir(category_path):
            if audio_file.endswith('.wav'):
                y_cat, sr_cat = librosa.load(os.path.join(category_path, audio_file), sr=None)
                mfcc_tmp = librosa.feature.mfcc(y=y_cat, sr=sr_cat, n_mfcc=13)
                mfccs_for_cat.append(np.mean(mfcc_tmp, axis=1))
        avg_mfccs.append(np.mean(mfccs_for_cat, axis=0))

    plt.figure(figsize=(10, 6))
    for i, avg_mfcc in enumerate(avg_mfccs):
        plt.plot(avg_mfcc, label=categories[i])
    plt.title('Average MFCCs Across Categories')
    plt.xlabel('MFCC Coefficients')
    plt.ylabel('Average Value')
    plt.legend()
    plt.show()



In [ ]:
if RUN_LEGACY_NOTEBOOK:

    ##############################################################################
    # 14) AMPLITUDE OVER TIME (SUPERIMPOSED WAVEFORMS)
    ##############################################################################
    # We'll plot one sample from each category on the same figure.
    print("\n=== Amplitude Over Time (One sample per category) ===")
    plt.figure(figsize=(10, 5))
    for category in categories:
        audio_path = os.path.join(dataset_path, category, os.listdir(os.path.join(dataset_path, category))[0])
        y_cat, sr_cat = librosa.load(audio_path, sr=None)
        plt.plot(y_cat, label=category)

    plt.title('Amplitude Over Time')
    plt.xlabel('Time')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 15) PCA VISUALIZATION OF AUDIO FEATURES
    ##############################################################################
    # We'll extract MFCC (mean) for each file, then do a 2D PCA, color-coded by category.
    print("\n=== PCA Visualization of Audio Features ===")
    pca_features = []
    pca_labels = []

    for category in categories:
        category_path = os.path.join(dataset_path, category)
        for audio_file in os.listdir(category_path):
            if audio_file.endswith('.wav'):
                y_pca, sr_pca = librosa.load(os.path.join(category_path, audio_file), sr=None)
                mfcc_pca = librosa.feature.mfcc(y=y_pca, sr=sr_pca, n_mfcc=13)
                pca_features.append(np.mean(mfcc_pca, axis=1))
                pca_labels.append(category)

    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(pca_features)

    df_pca = pd.DataFrame(pca_result, columns=['PC1', 'PC2'])
    df_pca['Category'] = pca_labels

    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='Category', s=100)
    plt.title('PCA Visualization of Audio Features')
    plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:

    ##############################################################################
    # 16) DISTRIBUTION OF SPECTRAL CENTROIDS
    ##############################################################################
    # We'll compute the spectral centroid for each file, store, and plot a histogram.
    print("\n=== Distribution of Spectral Centroids ===")
    centroids = []
    for category in categories:
        category_path = os.path.join(dataset_path, category)
        for audio_file in os.listdir(category_path):
            if audio_file.endswith('.wav'):
                y_cent, sr_cent = librosa.load(os.path.join(category_path, audio_file), sr=None)
                centroid = librosa.feature.spectral_centroid(y=y_cent, sr=sr_cent)[0]
                centroids.append(np.mean(centroid))

    sns.histplot(centroids, bins=20, kde=True)
    plt.title('Distribution of Spectral Centroids')
    plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 17) OPTIONAL: AVERAGE FREQUENCY BAND ENERGY (FROM THE ORIGINAL CODE)
    ##############################################################################
    # This was in the original script. We'll keep it to avoid removing anything.
    print("\n=== Average Frequency Band Energy by Category ===")
    band_energies = []
    for category in categories:
        category_path = os.path.join(dataset_path, category)
        energies = []
        for audio_file in os.listdir(category_path):
            if audio_file.endswith('.wav'):
                y_be, sr_be = librosa.load(os.path.join(category_path, audio_file), sr=None)
                stft = np.abs(librosa.stft(y_be))
                band_energy = np.sum(stft, axis=1)
                energies.append(np.mean(band_energy))
        band_energies.append(np.mean(energies))

    plt.bar(categories, band_energies)
    plt.title('Average Frequency Band Energy by Category')
    plt.ylabel('Energy')
    plt.show()



In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # Fix 1) Rename local augmentation functions to avoid confusion
    ##############################################################################
    def my_add_noise(audio, noise_factor=0.005):
        """Add random noise to the audio signal."""
        noise = np.random.randn(len(audio))
        return audio + noise_factor * noise

    def my_time_stretch(audio, rate=1.0):
        """Time-stretch audio by a certain rate, using librosa's built-in function."""
        # Note: we explicitly call librosa.effects.time_stretch with y= and rate=
        return librosa.effects.time_stretch(y=audio, rate=rate)

    def my_pitch_shift(audio, sr, n_steps=0):
        """Pitch-shift audio by n_steps semitones, using librosa's built-in function."""
        return librosa.effects.pitch_shift(y=audio, sr=sr, n_steps=n_steps)

    def augment_audio(y, sr):
        """
        Randomly apply one or more transformations to the audio
        to improve dataset variability.
        """
        import random
        choice = random.choice(['noise', 'stretch', 'pitch', 'none'])

        if choice == 'noise':
            return my_add_noise(y)
        elif choice == 'stretch':
            rate = random.uniform(0.8, 1.2)   # random stretch factor
            return my_time_stretch(y, rate=rate)
        elif choice == 'pitch':
            steps = random.uniform(-2, 2)     # random pitch shift in semitones
            return my_pitch_shift(y, sr, n_steps=steps)
        else:
            return y  # no augmentation


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    #############################################################################
    # NEW: DATA AUGMENTATION + HANDLING CLASS IMBALANCE
    # We'll find the largest category count and oversample the others by
    # generating augmented audio until each category matches the largest count.
    ##############################################################################
    print("\n=== Data Augmentation and Class Balancing ===")
    max_count = max(category_counts.values())

    all_audio_data = []   # Will store (audio_array, sr, label)
    for category in categories:
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        current_count = len(audio_files)

        # Load original audio
        loaded_signals = []
        for audio_file in audio_files:
            audio_path = os.path.join(category_path, audio_file)
            y_load, sr_load = librosa.load(audio_path, sr=None)
            loaded_signals.append((y_load, sr_load))
            all_audio_data.append((y_load, sr_load, category))

        # If category is underrepresented, generate augmented samples
        if current_count < max_count:
            needed = max_count - current_count
            # Randomly augment from existing signals
            for _ in range(needed):
                # pick a random original signal
                orig_y, orig_sr = random.choice(loaded_signals)
                # apply augmentation
                y_aug = augment_audio(orig_y, orig_sr)
                all_audio_data.append((y_aug, orig_sr, category))

    # Now we have an augmented list "all_audio_data"
    # with all categories having the same number of samples.


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 18) FEATURE EXTRACTION FOR MODEL TRAINING (MFCC MEAN FEATURES)
    ##############################################################################
    # We'll now extract features for the classification model.
    print("\n=== Preparing Features and Labels for Model ===")
    feature_list = []
    labels_list = []

    for category in categories:
        category_path = os.path.join(dataset_path, category)
        audio_files = [f for f in os.listdir(category_path) if f.endswith('.wav')]
        for audio_file in audio_files:
            audio_path = os.path.join(category_path, audio_file)
            y_data, sr_data = librosa.load(audio_path, sr=None)
            # Extract MFCC (13)
            mfcc_data = librosa.feature.mfcc(y=y_data, sr=sr_data, n_mfcc=13)
            # Take mean across time frames
            feature_list.append(np.mean(mfcc_data, axis=1))
            labels_list.append(category)

    X = np.array(feature_list)
    y = np.array(labels_list)

    print("Total Samples:", len(feature_list))
    print("Feature Shape:", X.shape)


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 19) LABEL ENCODING
    ##############################################################################
    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y)
    print("Labels Shape:", y_encoded.shape)


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 20) TRAIN-TEST SPLIT (80/20) AS REQUESTED
    ##############################################################################
    # We will specifically do 80/20, as per your preference (test_size=0.2).
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=42
    )

    print("Training Samples:", len(X_train))
    print("Testing Samples:", len(X_test))


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 21) RESHAPE FOR LSTM (SAMPLES, TIMESTEPS, 1)
    ##############################################################################
    # LSTM expects 3D input: (batch_size, timesteps, channels).
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test  = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 22) BUILD AND COMPILE THE LSTM MODEL (RETENTION OF ORIGINAL STRUCTURE)
    ##############################################################################
    model = Sequential()

    model.add(
        Conv1D(
            filters=32, kernel_size=3, activation='relu',
            input_shape=(X_train.shape[1], 1)
        )
    )
    model.add(MaxPooling1D(pool_size=2))
    model.add(LSTM(64, return_sequences=True))
    model.add(LSTM(64))
    model.add(Dropout(0.3))
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dense(len(np.unique(y_encoded)), activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.summary()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 23) TRAIN THE MODEL (SHOW TRAINING & VALIDATION ACCURACY)
    ##############################################################################
    history = model.fit(
        X_train,
        y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.2
    )


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 24) PLOT TRAIN & VALIDATION ACCURACY (Preference #16)
    ##############################################################################
    plt.figure(figsize=(8, 6))
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Train vs. Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 25) EVALUATE ON TEST SET
    ##############################################################################
    test_loss, test_acc = model.evaluate(X_test, y_test)
    print(f"Test Accuracy: {test_acc * 100:.2f}%")


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 26) CONFUSION MATRIX (WITH WHITE BACKGROUND)
    ##############################################################################
    y_pred_proba = model.predict(X_test)
    y_pred = np.argmax(y_pred_proba, axis=1)

    # Create confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    # Plot confusion matrix on a white background
    plt.figure(figsize=(6, 5))
    # Force the background to be white
    plt.rcParams['figure.facecolor'] = 'w'
    plt.rcParams['axes.facecolor'] = 'w'

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title("Confusion Matrix (Test Data)", color='black')
    plt.xlabel("Predicted Label", color='black')
    plt.ylabel("True Label", color='black')
    plt.show()

    # Print classification report as well
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=encoder.classes_))


In [ ]:
if RUN_LEGACY_NOTEBOOK:
    ##############################################################################
    # 27) AUC-ROC (WITH WHITE BACKGROUND), MULTI-CLASS (ONE-VS-REST)
    ##############################################################################
    # For multi-class, we do a one-vs-rest approach:
    n_classes = len(np.unique(y_test))
    y_test_binarized = label_binarize(y_test, classes=list(range(n_classes)))

    fpr = {}
    tpr = {}
    roc_auc = {}

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_pred_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Plot each class's ROC
    plt.figure(figsize=(6,5))
    plt.rcParams['figure.facecolor'] = 'w'
    plt.rcParams['axes.facecolor'] = 'w'

    colors = ['red','green','blue','orange','purple']  # or any distinct set
    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], label=f'Class {encoder.classes_[i]} (AUC = {roc_auc[i]:.2f})')

    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', color='black')
    plt.ylabel('True Positive Rate', color='black')
    plt.title('Multi-Class AUC-ROC', color='black')
    plt.legend(loc="lower right")
    plt.show()
